# Handwritten Formula to LaTeX OCR - Experiments

This notebook contains experiments for fine-tuning Vision-Language Models for handwritten formula recognition.

## 1. Setup

In [ ]:
# Install dependencies (if needed)
# !pip install -q transformers datasets peft accelerate bitsandbytes

import sys
sys.path.append('../src')

import torch
from datasets import load_dataset
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, Latex

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Dataset Exploration

In [ ]:
# Load LaTeX_OCR dataset
ds_latex_ocr = load_dataset("linxy/LaTeX_OCR", "human_handwrite")
print("LaTeX_OCR dataset:")
print(ds_latex_ocr)

In [ ]:
# Explore train split
train_ds = ds_latex_ocr['train']
print(f"\nTrain samples: {len(train_ds)}")
print(f"Columns: {train_ds.column_names}")
print(f"\nFirst example:")
print(train_ds[0])

In [ ]:
# Visualize some examples
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for idx, ax in enumerate(axes.flat):
    example = train_ds[idx]
    image = example['image']
    latex = example.get('text') or example.get('latex', '')
    
    ax.imshow(image)
    ax.set_title(f"LaTeX: {latex[:40]}..." if len(latex) > 40 else f"LaTeX: {latex}")
    ax.axis('off')

plt.tight_layout()
plt.savefig('../report/images/dataset_examples.png', dpi=150)
plt.show()

In [ ]:
# Load MathWriting dataset
ds_mathwriting = load_dataset("deepcopy/MathWriting-human")
print("MathWriting dataset:")
print(ds_mathwriting)

In [ ]:
# Analyze LaTeX formula lengths
import numpy as np

latex_lengths = [len(ex.get('text', '') or ex.get('latex', '')) for ex in train_ds]

plt.figure(figsize=(10, 4))
plt.hist(latex_lengths, bins=50, edgecolor='black')
plt.xlabel('LaTeX Length (characters)')
plt.ylabel('Count')
plt.title('Distribution of LaTeX Formula Lengths')
plt.axvline(np.mean(latex_lengths), color='r', linestyle='--', label=f'Mean: {np.mean(latex_lengths):.1f}')
plt.legend()
plt.savefig('../report/images/latex_length_distribution.png', dpi=150)
plt.show()

print(f"Mean length: {np.mean(latex_lengths):.1f}")
print(f"Max length: {max(latex_lengths)}")
print(f"Min length: {min(latex_lengths)}")

## 3. Model Loading

In [ ]:
from src.model_utils import ModelConfig, load_model_and_processor, VLMForLatexOCR

# Load model for inference
model_name = "HuggingFaceTB/SmolVLM-256M-Instruct"

vlm = VLMForLatexOCR.from_pretrained(
    model_name,
    load_in_4bit=False
)
print("Model loaded successfully!")

## 4. Zero-shot Inference

In [ ]:
# Test zero-shot on a few examples
test_ds = ds_latex_ocr['test']

print("Zero-shot inference results:")
print("=" * 60)

for i in range(5):
    example = test_ds[i]
    image = example['image']
    reference = example.get('text') or example.get('latex', '')
    
    prediction = vlm.generate(
        image=image,
        prompt="Convert this handwritten formula to LaTeX:",
        max_new_tokens=256
    )
    
    print(f"\nExample {i+1}:")
    print(f"Reference:  {reference}")
    print(f"Prediction: {prediction}")
    
    # Display image
    plt.figure(figsize=(6, 2))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"Example {i+1}")
    plt.show()

## 5. One-shot Inference

In [ ]:
# Test one-shot inference
print("One-shot inference results:")
print("=" * 60)

for i in range(5):
    example = test_ds[i]
    image = example['image']
    reference = example.get('text') or example.get('latex', '')
    
    prediction = vlm.generate_with_one_shot(
        image=image,
        example_latex="x^2 + y^2 = z^2",
        max_new_tokens=256
    )
    
    print(f"\nExample {i+1}:")
    print(f"Reference:  {reference}")
    print(f"Prediction: {prediction}")

## 6. Training

In [ ]:
# Training configuration
from src.train import TrainConfig, train

config = TrainConfig(
    model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    primary_subset="human_handwrite",
    use_secondary=False,
    num_epochs=3,
    batch_size=4,
    learning_rate=2e-4,
    use_lora=True,
    lora_r=16,
    load_in_4bit=True,
)

print("Training configuration:")
for key, value in config.__dict__.items():
    print(f"  {key}: {value}")

In [ ]:
# Run training (uncomment to actually train)
# model, processor = train(config, run_name="notebook_experiment")

## 7. Evaluation

In [ ]:
from src.evaluate import EvalConfig, run_full_evaluation
from src.metrics import compute_all_metrics, MetricTracker

# Quick evaluation on a subset
tracker = MetricTracker()

for i in range(10):
    example = test_ds[i]
    image = example['image']
    reference = example.get('text') or example.get('latex', '')
    
    prediction = vlm.generate(image=image)
    tracker.add(prediction, reference)

print("Evaluation Results (10 samples):")
print(tracker)

## 8. Results Visualization

In [ ]:
# Placeholder for results visualization
# Fill in after running full evaluation

results = {
    "zero_shot": {"bleu": 0.0, "exact_match": 0.0, "edit_distance": 0.0, "token_f1": 0.0},
    "one_shot": {"bleu": 0.0, "exact_match": 0.0, "edit_distance": 0.0, "token_f1": 0.0},
    "sft_latex_ocr": {"bleu": 0.0, "exact_match": 0.0, "edit_distance": 0.0, "token_f1": 0.0},
    "sft_combined": {"bleu": 0.0, "exact_match": 0.0, "edit_distance": 0.0, "token_f1": 0.0},
}

# Plot results
import pandas as pd

df = pd.DataFrame(results).T

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

metrics = ['bleu', 'exact_match', 'edit_distance', 'token_f1']
titles = ['BLEU Score', 'Exact Match', 'Edit Distance', 'Token F1']

for ax, metric, title in zip(axes, metrics, titles):
    df[metric].plot(kind='bar', ax=ax, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../report/images/evaluation_results.png', dpi=150)
plt.show()

## 9. Error Analysis

In [ ]:
# Analyze common errors
from src.metrics import normalize_latex, tokenize_latex

errors = []

for i in range(min(20, len(test_ds))):
    example = test_ds[i]
    image = example['image']
    reference = example.get('text') or example.get('latex', '')
    
    prediction = vlm.generate(image=image)
    
    if normalize_latex(prediction) != normalize_latex(reference):
        errors.append({
            'index': i,
            'reference': reference,
            'prediction': prediction,
            'image': image
        })

print(f"Errors found: {len(errors)}/20")

# Show first few errors
for error in errors[:5]:
    print(f"\nError #{error['index']}:")
    print(f"  Reference:  {error['reference']}")
    print(f"  Prediction: {error['prediction']}")
    plt.figure(figsize=(6, 2))
    plt.imshow(error['image'])
    plt.axis('off')
    plt.show()

## 10. Save Results

In [ ]:
# Save evaluation results
import json

with open('../evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to evaluation_results.json")